In [6]:
# ============================================================
# FULL, SELF-CONTAINED COLAB CELL:
#   - downloads SPARC Rotmod_LTG.zip
#   - defines solver + MCMC
#   - runs VSU(AQUAL axisymmetric) vs GR+NFW for one galaxy
# ============================================================

# --- deps (Colab usually has these) ---
# If SciPy is missing, uncomment:
# !pip -q install numpy scipy

import os, glob
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Optional, Callable
from scipy import optimize, interpolate, special

# -------------------------
# Download SPARC rotmod bundle
# -------------------------
ZIP_URL = "https://astroweb.case.edu/SPARC/Rotmod_LTG.zip"
ZIP_NAME = "Rotmod_LTG.zip"
OUT_DIR = "Rotmod_LTG"

if not os.path.exists(ZIP_NAME):
    !wget -q --show-progress -O {ZIP_NAME} {ZIP_URL}
else:
    print(f"{ZIP_NAME} already exists; skipping download.")

os.makedirs(OUT_DIR, exist_ok=True)
!unzip -oq {ZIP_NAME} -d {OUT_DIR}

rotmod_files = sorted(glob.glob(os.path.join(OUT_DIR, "*_rotmod.dat")))
assert len(rotmod_files) > 0, "No *_rotmod.dat files found after unzip."

# Prefer a common test galaxy; else fall back to the first file.
preferred = os.path.join(OUT_DIR, "NGC2403_rotmod.dat")
galaxy_path = preferred if os.path.exists(preferred) else rotmod_files[0]
print("Using galaxy file:", os.path.basename(galaxy_path))

# -------------------------
# Constants / units
# -------------------------
G = 4.30091e-6  # kpc (km/s)^2 / Msun
KPC_IN_M = 3.085677581e19
A0_SI = 1.2e-10  # m/s^2
A0 = A0_SI / (1e6 / KPC_IN_M)  # (km/s)^2/kpc

def mu_exponential(x: np.ndarray) -> np.ndarray:
    return 1.0 - np.exp(-x)

# -------------------------
# Rotmod loader
# -------------------------
@dataclass
class RotmodData:
    R: np.ndarray
    Vobs: np.ndarray
    eV: np.ndarray
    Vgas: np.ndarray
    Vdisk: np.ndarray
    Vbul: np.ndarray
    extra: Optional[np.ndarray] = None

def load_rotmod(path: str) -> RotmodData:
    arr = np.genfromtxt(path)
    if arr.ndim != 2 or arr.shape[1] < 6:
        raise ValueError(f"Unexpected rotmod shape {arr.shape} in {path}")
    R, Vobs, eV, Vgas, Vdisk, Vbul = arr[:,0], arr[:,1], arr[:,2], arr[:,3], arr[:,4], arr[:,5]
    extra = arr[:,6:] if arr.shape[1] > 6 else None
    # basic cleanup: drop NaNs
    m = np.isfinite(R) & np.isfinite(Vobs) & np.isfinite(eV) & np.isfinite(Vgas) & np.isfinite(Vdisk) & np.isfinite(Vbul)
    return RotmodData(R=R[m], Vobs=Vobs[m], eV=eV[m], Vgas=Vgas[m], Vdisk=Vdisk[m], Vbul=Vbul[m], extra=(extra[m] if extra is not None else None))

# -------------------------
# Exponential disk (thin) + fit
# -------------------------
def vdisk_exponential(R: np.ndarray, Mdisk: float, Rd: float) -> np.ndarray:
    R = np.asarray(R)
    y = R / (2.0*Rd + 1e-30)
    Sigma0 = Mdisk / (2.0*np.pi*Rd**2)
    I0, I1 = special.iv(0, y), special.iv(1, y)
    K0, K1 = special.kv(0, y), special.kv(1, y)
    term = I0*K0 - I1*K1
    V2 = 4.0*np.pi*G*Sigma0*Rd*(y**2)*term
    return np.sqrt(np.clip(V2, 0.0, np.inf))

def fit_exponential_disk(R: np.ndarray, V: np.ndarray, w: Optional[np.ndarray]=None) -> Tuple[float,float]:
    R = np.asarray(R); V = np.asarray(V)
    if w is None:
        w = np.ones_like(R)
    Rd0 = max(0.5, 0.5*np.median(R))
    M0 = 1e10
    def resid(p):
        logM, logRd = p
        M = 10**logM
        Rd = 10**logRd
        Vmod = vdisk_exponential(R, M, Rd)
        return (Vmod - V) * np.sqrt(w)
    p0 = np.array([np.log10(M0), np.log10(Rd0)])
    sol = optimize.least_squares(resid, p0, bounds=([6, -1],[13, 2]))
    return 10**sol.x[0], 10**sol.x[1]

# -------------------------
# Axisymmetric density builders
# -------------------------
def rho_exp_disk(R: np.ndarray, z: np.ndarray, Mdisk: float, Rd: float, hz: float) -> np.ndarray:
    Sigma0 = Mdisk / (2.0*np.pi*Rd**2)
    SigmaR = Sigma0 * np.exp(-R/Rd)
    return SigmaR * np.exp(-np.abs(z)/hz) / (2.0*hz + 1e-30)

# -------------------------
# Axisymmetric AQUAL solver (starter Picard + SOR)
# -------------------------
@dataclass
class Grid:
    R: np.ndarray
    z: np.ndarray
    dR: float
    dz: float

def make_grid(Rmax: float, zmax: float, Nr: int, Nz: int) -> Grid:
    R = np.linspace(0.0, Rmax, Nr)
    z = np.linspace(0.0, zmax, Nz)
    return Grid(R=R, z=z, dR=R[1]-R[0], dz=z[1]-z[0])

def boundary_phi_pointmass(R: float, z: float, Mtot: float, a0: float=A0) -> float:
    r = np.sqrt(R*R + z*z) + 1e-6
    return -np.sqrt(G*Mtot*a0) * np.log(r)

def solve_aqual_axisymmetric(
    grid: Grid,
    rho: np.ndarray,
    mu: Callable[[np.ndarray], np.ndarray] = mu_exponential,
    a0: float = A0,
    Mtot_for_bc: Optional[float] = None,
    max_outer: int = 20,
    max_inner: int = 120,
    omega: float = 1.6,
    tol: float = 2e-5,
) -> np.ndarray:
    Nr, Nz = rho.shape
    dR, dz = grid.dR, grid.dz

    if Mtot_for_bc is None:
        RR, ZZ = np.meshgrid(grid.R, grid.z, indexing='ij')
        vol = 2.0 * (2.0*np.pi*RR) * dR * dz  # factor 2 for z<0
        Mtot_for_bc = float(np.sum(rho * vol))

    Phi = np.zeros((Nr, Nz), dtype=float)
    for i in range(Nr):
        for j in range(Nz):
            Phi[i,j] = boundary_phi_pointmass(grid.R[i], grid.z[j], Mtot_for_bc, a0=a0)

    RHS = 4.0*np.pi*G*rho  # (km/s)^2/kpc^2

    def grad_mag(P):
        dPR = np.zeros_like(P)
        dPz = np.zeros_like(P)
        dPR[1:-1,:] = (P[2:,:] - P[:-2,:])/(2*dR)
        dPz[:,1:-1] = (P[:,2:] - P[:,:-2])/(2*dz)
        dPR[0,:] = 0.0
        dPz[:,0] = 0.0
        dPR[-1,:] = (P[-1,:] - P[-2,:])/dR
        dPz[:,-1] = (P[:,-1] - P[:,-2])/dz
        return np.sqrt(dPR*dPR + dPz*dPz) + 1e-30

    def apply_dirichlet(P):
        i = Nr-1
        for j in range(Nz):
            P[i,j] = boundary_phi_pointmass(grid.R[i], grid.z[j], Mtot_for_bc, a0=a0)
        j = Nz-1
        for i in range(Nr):
            P[i,j] = boundary_phi_pointmass(grid.R[i], grid.z[j], Mtot_for_bc, a0=a0)

    apply_dirichlet(Phi)

    for _outer in range(max_outer):
        gmag = grad_mag(Phi)
        mu_c = mu(gmag / a0)
        mu_Rp = 0.5*(mu_c[1:,:] + mu_c[:-1,:])  # (Nr-1,Nz)
        mu_Zp = 0.5*(mu_c[:,1:] + mu_c[:,:-1])  # (Nr,Nz-1)

        max_update = 0.0
        for _inner in range(max_inner):
            max_update = 0.0
            for i in range(Nr):
                Ri = grid.R[i]
                for j in range(Nz):
                    if i == Nr-1 or j == Nz-1:
                        continue

                    ip = i+1
                    im = i-1 if i-1 >= 0 else 1
                    jp = j+1
                    jm = j-1 if j-1 >= 0 else 1

                    mu_imh = mu_Rp[i-1,j] if i > 0 else mu_Rp[0,j]
                    mu_iph = mu_Rp[i,j]   if i < Nr-1 else mu_Rp[-1,j]

                    R_imh = max(0.0, Ri - 0.5*dR)
                    R_iph = Ri + 0.5*dR

                    aR_p = (R_iph * mu_iph) / (Ri * dR*dR + 1e-30)
                    aR_m = (R_imh * mu_imh) / (Ri * dR*dR + 1e-30) if Ri > 0 else aR_p

                    mu_jmh = mu_Zp[i,j-1] if j > 0 else mu_Zp[i,0]
                    mu_jph = mu_Zp[i,j]   if j < Nz-1 else mu_Zp[i,-1]
                    aZ_p = mu_jph / (dz*dz)
                    aZ_m = mu_jmh / (dz*dz)

                    aC = aR_p + aR_m + aZ_p + aZ_m
                    b = RHS[i,j]

                    Phi_new = (aR_p*Phi[ip,j] + aR_m*Phi[im,j] + aZ_p*Phi[i,jp] + aZ_m*Phi[i,jm] - b) / (aC + 1e-30)
                    upd = Phi_new - Phi[i,j]
                    Phi[i,j] += omega * upd
                    if abs(upd) > max_update:
                        max_update = abs(upd)

            apply_dirichlet(Phi)
            if max_update < tol:
                break

        if max_update < tol:
            break

    return Phi

def rotation_curve_from_phi(grid: Grid, Phi: np.ndarray, R_eval: np.ndarray) -> np.ndarray:
    Phi_mid = Phi[:,0]
    dR = grid.dR
    dPhidR = np.zeros_like(Phi_mid)
    dPhidR[1:-1] = (Phi_mid[2:] - Phi_mid[:-2])/(2*dR)
    dPhidR[0] = 0.0
    dPhidR[-1] = (Phi_mid[-1] - Phi_mid[-2])/dR
    f = interpolate.interp1d(grid.R, dPhidR, kind="linear", fill_value="extrapolate")
    gR = f(R_eval)
    return np.sqrt(np.clip(R_eval * gR, 0.0, np.inf))

# -------------------------
# MCMC
# -------------------------
def loglike_gaussian(y, yerr, ymodel) -> float:
    r = (y - ymodel)/yerr
    return -0.5*np.sum(r*r + np.log(2*np.pi*yerr*yerr))

def metropolis(logpost, x0, step, nsamp=8000, burn=2000, thin=10, seed=0):
    rng = np.random.default_rng(seed)
    x = np.array(x0, dtype=float)
    lp = float(logpost(x))
    chain = []
    acc = 0
    for t in range(nsamp):
        prop = x + rng.normal(scale=step, size=x.shape)
        lp_prop = float(logpost(prop))
        if np.log(rng.random()) < lp_prop - lp:
            x, lp = prop, lp_prop
            acc += 1
        if t >= burn and ((t-burn) % thin == 0):
            chain.append(x.copy())
    chain = np.asarray(chain)
    print(f"Metropolis acceptance ~ {acc/nsamp:.3f} | saved {chain.shape[0]} samples")
    return chain

# -------------------------
# NFW halo (real)
# -------------------------
def rho_crit_Msun_kpc3(H0_km_s_Mpc: float = 70.0) -> float:
    H0 = H0_km_s_Mpc / 1000.0  # km/s/kpc
    return 3.0 * H0*H0 / (8.0*np.pi*G)

def vhalo_nfw(R_kpc: np.ndarray, M200: float, c200: float, H0_km_s_Mpc: float = 70.0) -> np.ndarray:
    R = np.asarray(R_kpc)
    rc = rho_crit_Msun_kpc3(H0_km_s_Mpc)
    R200 = (3.0*M200 / (4.0*np.pi*200.0*rc))**(1.0/3.0)  # kpc
    rs = R200 / (c200 + 1e-30)
    def f(u): return np.log(1.0+u) - u/(1.0+u)
    x = R / (rs + 1e-30)
    fc = f(c200)
    Menc = M200 * f(x) / (fc + 1e-30)
    V2 = G * Menc / (R + 1e-30)
    return np.sqrt(np.clip(V2, 0.0, np.inf))

# -------------------------
# VSU interpolator build (precompute in Υ*)
# -------------------------
def build_vsu_interpolator_for_galaxy(
    rot: RotmodData,
    Rmax_factor: float = 3.0,
    zmax_factor: float = 2.0,
    Nr: int = 96,
    Nz: int = 64,
    hz_over_Rd: float = 0.2,
    ups_grid: np.ndarray = np.linspace(0.1, 1.0, 9),
    a0: float = A0,
):
    R_obs = rot.R

    # Fit exponential disks to the Newtonian component curves (fallback when you don't have Σ(R))
    Mgas, Rd_gas = fit_exponential_disk(R_obs, rot.Vgas)
    Mstar_unit, Rd_star = fit_exponential_disk(R_obs, rot.Vdisk)

    hz_star = hz_over_Rd * Rd_star
    hz_gas  = 0.1

    Rmax = float(Rmax_factor * R_obs.max())
    zmax = float(zmax_factor * R_obs.max())
    grid = make_grid(Rmax, zmax, Nr=Nr, Nz=Nz)
    RR, ZZ = np.meshgrid(grid.R, grid.z, indexing='ij')

    rho_gas = rho_exp_disk(RR, ZZ, Mgas, Rd_gas, hz_gas)

    Vgrid = []
    for k, ups in enumerate(ups_grid):
        rho_star = rho_exp_disk(RR, ZZ, ups*Mstar_unit, Rd_star, hz_star)
        rho = rho_gas + rho_star
        Phi = solve_aqual_axisymmetric(grid, rho, a0=a0)
        V = rotation_curve_from_phi(grid, Phi, R_obs)
        Vgrid.append(V)
        print(f"VSU precompute {k+1}/{len(ups_grid)}: Υ*={ups:.3f}")

    Vgrid = np.asarray(Vgrid)  # (Nups, Nobs)
    f = interpolate.interp1d(ups_grid, Vgrid, kind="cubic", axis=0, fill_value="extrapolate")
    return lambda ups: np.asarray(f(ups), dtype=float)

# -------------------------
# Per-galaxy fit
# -------------------------
def fit_galaxy_vsu_vs_nfw(rot: RotmodData):
    R = rot.R
    y = rot.Vobs
    yerr = rot.eV

    # --- VSU ---
    V_of_ups = build_vsu_interpolator_for_galaxy(rot)

    def logpost_vsu(x):
        ups = float(x[0])
        if not (0.01 < ups < 2.0):
            return -np.inf
        lp = -0.5*((np.log10(ups) - np.log10(0.5))/0.25)**2
        Vmod = V_of_ups(ups)
        return lp + loglike_gaussian(y, yerr, Vmod)

    chain_vsu = metropolis(logpost_vsu, x0=[0.5], step=[0.06], nsamp=7000, burn=1500, thin=10, seed=0)
    ups_vsu = chain_vsu[:,0]
    ups_mean = float(np.mean(ups_vsu))
    ups_std  = float(np.std(ups_vsu))

    # --- NFW ---
    def logpost_nfw(x):
        logM200, logc, ups = map(float, x)
        if not (9.0 < logM200 < 14.0 and 0.2 < logc < 1.5 and 0.01 < ups < 2.0):
            return -np.inf
        Vbar2 = rot.Vgas**2 + ups*rot.Vdisk**2 + ups*rot.Vbul**2
        Vhalo = vhalo_nfw(R, 10**logM200, 10**logc, H0_km_s_Mpc=70.0)
        Vmod = np.sqrt(np.clip(Vbar2 + Vhalo**2, 0.0, np.inf))
        return loglike_gaussian(y, yerr, Vmod)

    chain_nfw = metropolis(logpost_nfw, x0=[11.5, 0.9, 0.5], step=[0.09, 0.06, 0.06], nsamp=9000, burn=2000, thin=10, seed=1)
    ups_nfw = chain_nfw[:,2]
    ups_mean_nfw = float(np.mean(ups_nfw))
    ups_std_nfw  = float(np.std(ups_nfw))

    return dict(
        ups_vsu=ups_mean, sigma_ups_vsu=ups_std,
        ups_nfw=ups_mean_nfw, sigma_ups_nfw=ups_std_nfw,
        galaxy=os.path.basename(galaxy_path)
    )

# -------------------------
# RUN
# -------------------------
rot = load_rotmod(galaxy_path)
out = fit_galaxy_vsu_vs_nfw(rot)

print("\n================ RESULTS ================")
print("Galaxy:", out["galaxy"])
print(f"VSU: Υ* = {out['ups_vsu']:.3f} ± {out['sigma_ups_vsu']:.3f}")
print(f"NFW: Υ* = {out['ups_nfw']:.3f} ± {out['sigma_ups_nfw']:.3f}")
print("========================================")


Rotmod_LTG.zip already exists; skipping download.
Using galaxy file: NGC2403_rotmod.dat
VSU precompute 1/9: Υ*=0.100
VSU precompute 2/9: Υ*=0.213
VSU precompute 3/9: Υ*=0.325
VSU precompute 4/9: Υ*=0.438
VSU precompute 5/9: Υ*=0.550
VSU precompute 6/9: Υ*=0.662
VSU precompute 7/9: Υ*=0.775
VSU precompute 8/9: Υ*=0.887
VSU precompute 9/9: Υ*=1.000
Metropolis acceptance ~ 0.045 | saved 550 samples
Metropolis acceptance ~ 0.002 | saved 700 samples

================ RESULTS ================
Galaxy: NGC2403_rotmod.dat
VSU: Υ* = 0.689 ± 0.002
NFW: Υ* = 0.284 ± 0.030


In [1]:
# ============================================================
# FULL, SELF-CONTAINED COLAB CELL (v2):
#   - downloads SPARC Rotmod_LTG.zip
#   - defines VSU(AQUAL axisymmetric) solver + MCMC
#   - defines NFW halo + ADAPTIVE Metropolis (so NFW doesn't stall)
#   - runs VSU vs GR+NFW for one galaxy and prints results + diagnostics
# ============================================================

# If SciPy is missing in your Colab runtime, uncomment:
# !pip -q install numpy scipy

import os, glob
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Optional, Callable
from scipy import optimize, interpolate, special

# -------------------------
# Download SPARC rotmod bundle
# -------------------------
ZIP_URL = "https://astroweb.case.edu/SPARC/Rotmod_LTG.zip"
ZIP_NAME = "Rotmod_LTG.zip"
OUT_DIR = "Rotmod_LTG"

if not os.path.exists(ZIP_NAME):
    !wget -q --show-progress -O {ZIP_NAME} {ZIP_URL}
else:
    print(f"{ZIP_NAME} already exists; skipping download.")

os.makedirs(OUT_DIR, exist_ok=True)
!unzip -oq {ZIP_NAME} -d {OUT_DIR}

rotmod_files = sorted(glob.glob(os.path.join(OUT_DIR, "*_rotmod.dat")))
assert len(rotmod_files) > 0, "No *_rotmod.dat files found after unzip."

preferred = os.path.join(OUT_DIR, "NGC2403_rotmod.dat")
galaxy_path = preferred if os.path.exists(preferred) else rotmod_files[0]
print("Using galaxy file:", os.path.basename(galaxy_path))

# -------------------------
# Constants / units
# -------------------------
G = 4.30091e-6  # kpc (km/s)^2 / Msun
KPC_IN_M = 3.085677581e19
A0_SI = 1.2e-10  # m/s^2
A0 = A0_SI / (1e6 / KPC_IN_M)  # (km/s)^2/kpc

def mu_exponential(x: np.ndarray) -> np.ndarray:
    return 1.0 - np.exp(-x)

# -------------------------
# Rotmod loader
# -------------------------
@dataclass
class RotmodData:
    R: np.ndarray
    Vobs: np.ndarray
    eV: np.ndarray
    Vgas: np.ndarray
    Vdisk: np.ndarray
    Vbul: np.ndarray
    extra: Optional[np.ndarray] = None

def load_rotmod(path: str) -> RotmodData:
    arr = np.genfromtxt(path)
    if arr.ndim != 2 or arr.shape[1] < 6:
        raise ValueError(f"Unexpected rotmod shape {arr.shape} in {path}")
    R, Vobs, eV, Vgas, Vdisk, Vbul = arr[:,0], arr[:,1], arr[:,2], arr[:,3], arr[:,4], arr[:,5]
    extra = arr[:,6:] if arr.shape[1] > 6 else None
    m = np.isfinite(R) & np.isfinite(Vobs) & np.isfinite(eV) & np.isfinite(Vgas) & np.isfinite(Vdisk) & np.isfinite(Vbul)
    return RotmodData(
        R=R[m], Vobs=Vobs[m], eV=eV[m],
        Vgas=Vgas[m], Vdisk=Vdisk[m], Vbul=Vbul[m],
        extra=(extra[m] if extra is not None else None)
    )

# -------------------------
# Exponential disk (thin) + fit
# -------------------------
def vdisk_exponential(R: np.ndarray, Mdisk: float, Rd: float) -> np.ndarray:
    R = np.asarray(R)
    y = R / (2.0*Rd + 1e-30)
    Sigma0 = Mdisk / (2.0*np.pi*Rd**2)
    I0, I1 = special.iv(0, y), special.iv(1, y)
    K0, K1 = special.kv(0, y), special.kv(1, y)
    term = I0*K0 - I1*K1
    V2 = 4.0*np.pi*G*Sigma0*Rd*(y**2)*term
    return np.sqrt(np.clip(V2, 0.0, np.inf))

def fit_exponential_disk(R: np.ndarray, V: np.ndarray, w: Optional[np.ndarray]=None) -> Tuple[float,float]:
    R = np.asarray(R); V = np.asarray(V)
    if w is None:
        w = np.ones_like(R)
    Rd0 = max(0.5, 0.5*np.median(R))
    M0 = 1e10
    def resid(p):
        logM, logRd = p
        M = 10**logM
        Rd = 10**logRd
        Vmod = vdisk_exponential(R, M, Rd)
        return (Vmod - V) * np.sqrt(w)
    p0 = np.array([np.log10(M0), np.log10(Rd0)])
    sol = optimize.least_squares(resid, p0, bounds=([6, -1],[13, 2]))
    return 10**sol.x[0], 10**sol.x[1]

# -------------------------
# Axisymmetric density builders
# -------------------------
def rho_exp_disk(R: np.ndarray, z: np.ndarray, Mdisk: float, Rd: float, hz: float) -> np.ndarray:
    Sigma0 = Mdisk / (2.0*np.pi*Rd**2)
    SigmaR = Sigma0 * np.exp(-R/Rd)
    return SigmaR * np.exp(-np.abs(z)/hz) / (2.0*hz + 1e-30)

# -------------------------
# Axisymmetric AQUAL solver (Picard + SOR)
#   NOTE: This includes a stability fix at the symmetry axes.
# -------------------------
@dataclass
class Grid:
    R: np.ndarray
    z: np.ndarray
    dR: float
    dz: float

def make_grid(Rmax: float, zmax: float, Nr: int, Nz: int) -> Grid:
    R = np.linspace(0.0, Rmax, Nr)
    z = np.linspace(0.0, zmax, Nz)
    return Grid(R=R, z=z, dR=R[1]-R[0], dz=z[1]-z[0])

def boundary_phi_pointmass(R: float, z: float, Mtot: float, a0: float=A0) -> float:
    r = np.sqrt(R*R + z*z) + 1e-6
    return -np.sqrt(G*Mtot*a0) * np.log(r)

def solve_aqual_axisymmetric(
    grid: Grid,
    rho: np.ndarray,
    mu: Callable[[np.ndarray], np.ndarray] = mu_exponential,
    a0: float = A0,
    Mtot_for_bc: Optional[float] = None,
    max_outer: int = 20,
    max_inner: int = 120,
    omega: float = 1.6,
    tol: float = 2e-5,
) -> np.ndarray:
    Nr, Nz = rho.shape
    dR, dz = grid.dR, grid.dz

    if Mtot_for_bc is None:
        RR, ZZ = np.meshgrid(grid.R, grid.z, indexing='ij')
        vol = 2.0 * (2.0*np.pi*RR) * dR * dz
        Mtot_for_bc = float(np.sum(rho * vol))

    Phi = np.zeros((Nr, Nz), dtype=float)
    for i in range(Nr):
        for j in range(Nz):
            Phi[i,j] = boundary_phi_pointmass(grid.R[i], grid.z[j], Mtot_for_bc, a0=a0)

    RHS = 4.0*np.pi*G*rho

    def grad_mag(P):
        dPR = np.zeros_like(P)
        dPz = np.zeros_like(P)
        dPR[1:-1,:] = (P[2:,:] - P[:-2,:])/(2*dR)
        dPz[:,1:-1] = (P[:,2:] - P[:,:-2])/(2*dz)
        dPR[0,:] = 0.0
        dPz[:,0] = 0.0
        dPR[-1,:] = (P[-1,:] - P[-2,:])/dR
        dPz[:,-1] = (P[:,-1] - P[:,-2])/dz
        return np.sqrt(dPR*dPR + dPz*dPz) + 1e-30

    def apply_dirichlet(P):
        i = Nr-1
        for j in range(Nz):
            P[i,j] = boundary_phi_pointmass(grid.R[i], grid.z[j], Mtot_for_bc, a0=a0)
        j = Nz-1
        for i in range(Nr):
            P[i,j] = boundary_phi_pointmass(grid.R[i], grid.z[j], Mtot_for_bc, a0=a0)

    apply_dirichlet(Phi)

    for _outer in range(max_outer):
        gmag = grad_mag(Phi)
        mu_c = mu(gmag / a0)
        mu_Rp = 0.5*(mu_c[1:,:] + mu_c[:-1,:])
        mu_Zp = 0.5*(mu_c[:,1:] + mu_c[:,:-1])

        max_update = 0.0
        for _inner in range(max_inner):
            max_update = 0.0

            # stability: enforce symmetry axes each sweep
            Phi[0, :] = Phi[1, :]
            Phi[:, 0] = Phi[:, 1]

            for i in range(1, Nr-1):
                Ri = grid.R[i]
                for j in range(1, Nz-1):
                    ip = i+1
                    im = i-1
                    jp = j+1
                    jm = j-1

                    mu_imh = mu_Rp[i-1,j]
                    mu_iph = mu_Rp[i,j]

                    R_imh = Ri - 0.5*dR
                    R_iph = Ri + 0.5*dR

                    aR_p = (R_iph * mu_iph) / (Ri * dR*dR + 1e-30)
                    aR_m = (R_imh * mu_imh) / (Ri * dR*dR + 1e-30)

                    mu_jmh = mu_Zp[i,j-1]
                    mu_jph = mu_Zp[i,j]
                    aZ_p = mu_jph / (dz*dz)
                    aZ_m = mu_jmh / (dz*dz)

                    aC = aR_p + aR_m + aZ_p + aZ_m
                    b = RHS[i,j]

                    Phi_new = (aR_p*Phi[ip,j] + aR_m*Phi[im,j] + aZ_p*Phi[i,jp] + aZ_m*Phi[i,jm] - b) / (aC + 1e-30)
                    upd = Phi_new - Phi[i,j]
                    Phi[i,j] += omega * upd
                    if abs(upd) > max_update:
                        max_update = abs(upd)

            apply_dirichlet(Phi)
            if max_update < tol:
                break

        if max_update < tol:
            break

    return Phi

def rotation_curve_from_phi(grid: Grid, Phi: np.ndarray, R_eval: np.ndarray) -> np.ndarray:
    Phi_mid = Phi[:,0]
    dR = grid.dR
    dPhidR = np.zeros_like(Phi_mid)
    dPhidR[1:-1] = (Phi_mid[2:] - Phi_mid[:-2])/(2*dR)
    dPhidR[0] = (Phi_mid[1] - Phi_mid[0]) / dR
    dPhidR[-1] = (Phi_mid[-1] - Phi_mid[-2]) / dR
    f = interpolate.interp1d(grid.R, dPhidR, kind="linear", fill_value="extrapolate")
    gR = f(R_eval)
    return np.sqrt(np.clip(R_eval * gR, 0.0, np.inf))

# -------------------------
# Likelihood
# -------------------------
def loglike_gaussian(y, yerr, ymodel) -> float:
    r = (y - ymodel)/yerr
    return -0.5*np.sum(r*r + np.log(2*np.pi*yerr*yerr))

# -------------------------
# Metropolis (plain)
# -------------------------
def metropolis(logpost, x0, step, nsamp=8000, burn=2000, thin=10, seed=0):
    rng = np.random.default_rng(seed)
    x = np.array(x0, dtype=float)
    lp = float(logpost(x))
    chain = []
    acc = 0
    for t in range(nsamp):
        prop = x + rng.normal(scale=step, size=x.shape)
        lp_prop = float(logpost(prop))
        if np.log(rng.random()) < lp_prop - lp:
            x, lp = prop, lp_prop
            acc += 1
        if t >= burn and ((t-burn) % thin == 0):
            chain.append(x.copy())
    chain = np.asarray(chain)
    print(f"Metropolis acceptance ~ {acc/nsamp:.3f} | saved {chain.shape[0]} samples")
    return chain

# -------------------------
# Metropolis (adaptive, for NFW)
# -------------------------
def metropolis_adaptive(
    logpost,
    x0,
    step0,
    nsamp=30000,
    burn=12000,
    thin=20,
    seed=1,
    target_accept=0.25,
    adapt_every=200,
    adapt_strength=0.06,
):
    rng = np.random.default_rng(seed)
    x = np.array(x0, dtype=float)
    step = np.array(step0, dtype=float)
    lp = float(logpost(x))

    chain = []
    acc = 0
    win_acc = 0
    win_n = 0

    for t in range(nsamp):
        prop = x + rng.normal(scale=step, size=x.shape)
        lp_prop = float(logpost(prop))
        if np.log(rng.random()) < lp_prop - lp:
            x, lp = prop, lp_prop
            acc += 1
            win_acc += 1
        win_n += 1

        if t < burn and (t + 1) % adapt_every == 0:
            rate = win_acc / max(1, win_n)
            step *= np.exp(adapt_strength * (rate - target_accept))
            win_acc = 0
            win_n = 0

        if t >= burn and ((t - burn) % thin == 0):
            chain.append(x.copy())

    chain = np.asarray(chain)
    print(f"Adaptive Metropolis acceptance ~ {acc/nsamp:.3f} | saved {chain.shape[0]} samples | final step={step}")
    return chain

# -------------------------
# NFW halo (real)
# -------------------------
def rho_crit_Msun_kpc3(H0_km_s_Mpc: float = 70.0) -> float:
    H0 = H0_km_s_Mpc / 1000.0  # km/s/kpc
    return 3.0 * H0*H0 / (8.0*np.pi*G)

def vhalo_nfw(R_kpc: np.ndarray, M200: float, c200: float, H0_km_s_Mpc: float = 70.0) -> np.ndarray:
    R = np.asarray(R_kpc)
    rc = rho_crit_Msun_kpc3(H0_km_s_Mpc)
    R200 = (3.0*M200 / (4.0*np.pi*200.0*rc))**(1.0/3.0)
    rs = R200 / (c200 + 1e-30)
    def f(u): return np.log(1.0+u) - u/(1.0+u)
    x = R / (rs + 1e-30)
    fc = f(c200)
    Menc = M200 * f(x) / (fc + 1e-30)
    V2 = G * Menc / (R + 1e-30)
    return np.sqrt(np.clip(V2, 0.0, np.inf))

# -------------------------
# VSU interpolator build (precompute in Υ*)
# -------------------------
def build_vsu_interpolator_for_galaxy(
    rot: RotmodData,
    Rmax_factor: float = 3.0,
    zmax_factor: float = 2.0,
    Nr: int = 96,
    Nz: int = 64,
    hz_over_Rd: float = 0.2,
    ups_grid: np.ndarray = np.linspace(0.1, 1.0, 9),
    a0: float = A0,
):
    R_obs = rot.R

    Mgas, Rd_gas = fit_exponential_disk(R_obs, rot.Vgas)
    Mstar_unit, Rd_star = fit_exponential_disk(R_obs, rot.Vdisk)

    hz_star = hz_over_Rd * Rd_star
    hz_gas  = 0.1

    Rmax = float(Rmax_factor * R_obs.max())
    zmax = float(zmax_factor * R_obs.max())
    grid = make_grid(Rmax, zmax, Nr=Nr, Nz=Nz)
    RR, ZZ = np.meshgrid(grid.R, grid.z, indexing='ij')

    rho_gas = rho_exp_disk(RR, ZZ, Mgas, Rd_gas, hz_gas)

    Vgrid = []
    for k, ups in enumerate(ups_grid):
        rho_star = rho_exp_disk(RR, ZZ, ups*Mstar_unit, Rd_star, hz_star)
        rho = rho_gas + rho_star
        Phi = solve_aqual_axisymmetric(grid, rho, a0=a0)
        if not np.isfinite(Phi).all():
            raise RuntimeError(f"Non-finite Phi at ups={ups}")
        V = rotation_curve_from_phi(grid, Phi, R_obs)
        if (not np.isfinite(V).all()) or (V.max() > 800):
            raise RuntimeError(f"Bad V(R) at ups={ups}: min={V.min()}, max={V.max()}")
        Vgrid.append(V)
        print(f"VSU precompute {k+1}/{len(ups_grid)}: Υ*={ups:.3f}")

    Vgrid = np.asarray(Vgrid)
    f = interpolate.interp1d(ups_grid, Vgrid, kind="cubic", axis=0, fill_value="extrapolate")
    return lambda ups: np.asarray(f(ups), dtype=float)

# -------------------------
# Per-galaxy fit
# -------------------------
def fit_galaxy_vsu_vs_nfw(rot: RotmodData):
    R = rot.R
    y = rot.Vobs
    yerr = rot.eV

    # --- VSU ---
    V_of_ups = build_vsu_interpolator_for_galaxy(rot)

    def logpost_vsu(x):
        ups = float(x[0])
        if not (0.01 < ups < 2.0):
            return -np.inf
        lp = -0.5*((np.log10(ups) - np.log10(0.5))/0.25)**2
        Vmod = V_of_ups(ups)
        return lp + loglike_gaussian(y, yerr, Vmod)

    chain_vsu = metropolis(logpost_vsu, x0=[0.5], step=[0.06], nsamp=7000, burn=1500, thin=10, seed=0)
    ups_vsu = chain_vsu[:,0]
    ups_mean = float(np.mean(ups_vsu))
    ups_std  = float(np.std(ups_vsu))

    # --- NFW (adaptive so it actually mixes) ---
    def logpost_nfw(x):
        logM200, logc, ups = map(float, x)
        if not (9.0 < logM200 < 14.0 and 0.2 < logc < 1.5 and 0.01 < ups < 2.0):
            return -np.inf
        Vbar2 = rot.Vgas**2 + ups*rot.Vdisk**2 + ups*rot.Vbul**2
        Vhalo = vhalo_nfw(R, 10**logM200, 10**logc, H0_km_s_Mpc=70.0)
        Vmod = np.sqrt(np.clip(Vbar2 + Vhalo**2, 0.0, np.inf))
        return loglike_gaussian(y, yerr, Vmod)

    chain_nfw = metropolis_adaptive(
        logpost_nfw,
        x0=[11.5, 0.9, 0.5],
        step0=[0.08, 0.05, 0.05],
        nsamp=30000,
        burn=12000,
        thin=20,
        seed=1,
    )
    ups_nfw = chain_nfw[:,2]
    ups_mean_nfw = float(np.mean(ups_nfw))
    ups_std_nfw  = float(np.std(ups_nfw))

    # Diagnostics for whether NFW chain actually moved
    nfw_std = chain_nfw.std(axis=0)

    return dict(
        ups_vsu=ups_mean, sigma_ups_vsu=ups_std,
        ups_nfw=ups_mean_nfw, sigma_ups_nfw=ups_std_nfw,
        nfw_chain_std=nfw_std,
        galaxy=os.path.basename(galaxy_path)
    )

# -------------------------
# RUN
# -------------------------
rot = load_rotmod(galaxy_path)
out = fit_galaxy_vsu_vs_nfw(rot)

print("\n================ RESULTS ================")
print("Galaxy:", out["galaxy"])
print(f"VSU: Υ* = {out['ups_vsu']:.3f} ± {out['sigma_ups_vsu']:.3f}")
print(f"NFW: Υ* = {out['ups_nfw']:.3f} ± {out['sigma_ups_nfw']:.3f}")
print("NFW chain std [logM200, logc, ups] =", out["nfw_chain_std"])
print("========================================")


Rotmod_LTG.zip      100%[===================>] 108.14K  --.-KB/s    in 0.1s    
Using galaxy file: NGC2403_rotmod.dat
VSU precompute 1/9: Υ*=0.100
VSU precompute 2/9: Υ*=0.213
VSU precompute 3/9: Υ*=0.325
VSU precompute 4/9: Υ*=0.438
VSU precompute 5/9: Υ*=0.550
VSU precompute 6/9: Υ*=0.662
VSU precompute 7/9: Υ*=0.775
VSU precompute 8/9: Υ*=0.887
VSU precompute 9/9: Υ*=1.000
Metropolis acceptance ~ 0.041 | saved 550 samples
Adaptive Metropolis acceptance ~ 0.011 | saved 900 samples | final step=[0.03355637 0.02097273 0.02097273]

================ RESULTS ================
Galaxy: NGC2403_rotmod.dat
VSU: Υ* = 1.605 ± 0.002
NFW: Υ* = 0.283 ± 0.025
NFW chain std [logM200, logc, ups] = [0.01210158 0.01569121 0.02505576]
